In [2]:
import os
import sys
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm
from torchvision import transforms
import timm

c:\Users\minhp\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# =========================================
# IMPORT DATASET
# =========================================
sys.path.append(os.path.abspath(".."))
from scripts.Loading_Dataset import LiverDataset

In [4]:
# =========================================
# DEVICE
# =========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [5]:
# =========================================
# TRANSFORMS (IMPORTANT FIX)
# PIL → Tensor → ViT compatible
# =========================================
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

In [6]:
# =========================================
# DATASET
# =========================================
root_dir = "../Dataset"

dataset = LiverDataset(
    root_dir=root_dir,
    transform=transform
)


In [7]:
# =========================================
# DATALOADER
# =========================================
dataloader = DataLoader(
    dataset,
    batch_size=32,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

In [8]:
# =========================================
# VISION TRANSFORMER (ViT)
# =========================================
model = timm.create_model(
    "vit_base_patch16_224",
    pretrained=True,
    num_classes=0   # returns CLS embedding
)

model = model.to(device)
model.eval()

VisionTransformer(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
    (norm): Identity()
  )
  (pos_drop): Dropout(p=0.0, inplace=False)
  (patch_drop): Identity()
  (norm_pre): Identity()
  (blocks): Sequential(
    (0): Block(
      (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
      (attn): Attention(
        (qkv): Linear(in_features=768, out_features=2304, bias=True)
        (q_norm): Identity()
        (k_norm): Identity()
        (attn_drop): Dropout(p=0.0, inplace=False)
        (norm): Identity()
        (proj): Linear(in_features=768, out_features=768, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (ls1): Identity()
      (drop_path1): Identity()
      (norm2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
      (mlp): Mlp(
        (fc1): Linear(in_features=768, out_features=3072, bias=True)
        (act): GELU(approximate='none')
        (drop1): Dropout(p=0.0, inplace=False

In [9]:

# =========================================
# FEATURE EXTRACTION
# =========================================
all_features = []
all_labels = []

with torch.no_grad():
    for images, labels in tqdm(dataloader, desc="Extracting ViT features"):

        # move to GPU
        images = images.to(device)

        # forward pass → [B, 768]
        features = model(images)

        # store results
        all_features.append(features.cpu())
        all_labels.append(labels)

Extracting ViT features:   0%|          | 0/112 [00:00<?, ?it/s]c:\Users\minhp\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\utils\data\dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Extracting ViT features: 100%|██████████| 112/112 [35:41<00:00, 19.12s/it] 


In [10]:
# =========================================
# MERGE RESULTS
# =========================================
all_features = torch.cat(all_features, dim=0)
all_labels = torch.cat(all_labels, dim=0)

In [11]:
# =========================================
# OUTPUT
# =========================================
print("Feature shape:", all_features.shape)  # [N, 768]
print("Label shape:", all_labels.shape)

print("\nSample feature vector:")
print(all_features[0])

Feature shape: torch.Size([3557, 768])
Label shape: torch.Size([3557])

Sample feature vector:
tensor([-8.2186e-01,  8.1804e-01, -2.0194e+00,  6.6517e-01, -2.5191e+00,
         3.4186e-01, -8.7095e-01,  7.0768e-01,  2.2042e+00, -1.5675e-01,
         1.2619e+00,  1.4163e+00,  1.0925e-01, -1.0824e+00,  6.6334e-01,
        -7.3465e-01,  8.0054e-01,  1.8124e+00, -2.5448e-01,  6.7286e-01,
         1.0830e+00,  1.5056e+00,  5.1257e-01,  5.4740e-01, -8.1749e-01,
        -2.0935e+00,  9.1572e-01, -1.2964e+00, -1.3666e-01, -7.3786e-02,
        -4.5366e-01, -1.2830e+00,  2.6807e-01,  6.5699e-01,  1.8565e+00,
        -6.6025e-01, -1.4528e+00,  5.8057e-01, -3.0225e+00,  1.4928e+00,
         2.4847e+00,  6.4096e-01,  7.4960e-01,  5.9600e-01, -1.6151e+00,
         3.9917e-01,  2.3761e+00, -2.0592e+00, -1.0570e+00, -5.9477e-01,
        -2.1736e+00,  2.0091e+00, -1.0181e+00, -1.6005e+00, -1.4953e+00,
        -1.4060e+00, -1.4416e+00, -7.0132e-01,  1.8208e+00,  5.7668e+00,
        -8.2979e-01,  1.4070e

In [ ]:
#Numeric representation of image
torch.save(all_features, "features.pt")

#Saved as tensors that represents class ids of each image in dataset
torch.save(all_labels, "labels.pt")